In [1]:
import numpy as np
from tqdm import tqdm
import ot_utils
import itertools

file = np.load("all_jets_9_subjets_p4s.npz")

all_jets_transformed_p4s = file["X_transformed_epxpypz"]

In [2]:
all_jets_transformed_p4s.shape

(32000, 9, 4)

In [3]:
dataset_length = 4000
num_dataset = 8


def load_completed(path):
    """Read completed (i,j) pairs from progress file, return empty set if file missing."""
    try:
        with open(path, "r") as f:
            return {tuple(map(int, line.split(","))) for line in f if line.strip()}
    except FileNotFoundError:
        return set()


def save_progress(path, i, j):
    """Append a completed (i,j) pair to progress file."""
    with open(path, "a") as f:
        f.write(f"{i},{j}\n")


progress_file = "progress.txt"
completed = load_completed(progress_file)
total = num_dataset * (num_dataset + 1) // 2
pairs = itertools.combinations_with_replacement(range(num_dataset), 2)

with tqdm(total=total, desc="blocks") as pbar:
    for i, j in pairs:
        if (i, j) in completed:
            pbar.update()
            # print(i, j)
            continue

        # slice out your two event-blocks
        start_i = i * dataset_length
        end_i = (i + 1) * dataset_length
        start_j = j * dataset_length
        end_j = (j + 1) * dataset_length

        if i == j:
            block = ot_utils.parallel_dist_mat_square(
                event_set=all_jets_transformed_p4s[start_i:end_i],
                dist_func=ot_utils.PSdistance,
                events_per_process=8192,
                Sphere2Simplex=1,
                l_has_chi=True,
                return_only_l=True
            )
        else:
            block = ot_utils.parallel_dist_mat_non_square(
                all_jets_transformed_p4s[start_i:end_i],
                all_jets_transformed_p4s[start_j:end_j],
                ot_utils.PSdistance,
                events_per_process=8192,
                Sphere2Simplex=1,
                l_has_chi=True,
                return_only_l=True
            )

        # save and record progress
        np.save(f"ps_block_{i}_{j}.npy", block)
        save_progress(progress_file, i, j)
        pbar.update()

blocks: 100%|██████████| 36/36 [2:56:03<00:00, 293.43s/it]  


In [3]:
ps_mat = np.load("ps_block_0_1.npy")
print(ps_mat.shape)

(4000, 4000)
